**Imports**

In [1]:
import pandas as pd

**Loading dataset**

In [2]:
df_xray_raw = pd.read_csv("../../../data/raw/train.csv")

**Dataset description**

In [3]:
nb_lines = df_xray_raw.shape[0]
nb_columns = df_xray_raw.shape[1]

print("The data file contains :")
print(f"{nb_lines} lines and {nb_columns} columns")

The data file contains :
223414 lines and 19 columns


In [4]:
print(df_xray_raw.info())

<class 'pandas.DataFrame'>
RangeIndex: 223414 entries, 0 to 223413
Data columns (total 19 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Path                        223414 non-null  str    
 1   Sex                         223414 non-null  str    
 2   Age                         223414 non-null  int64  
 3   Frontal/Lateral             223414 non-null  str    
 4   AP/PA                       191027 non-null  str    
 5   No Finding                  22381 non-null   float64
 6   Enlarged Cardiomediastinum  44839 non-null   float64
 7   Cardiomegaly                46203 non-null   float64
 8   Lung Opacity                117778 non-null  float64
 9   Lung Lesion                 11944 non-null   float64
 10  Edema                       85956 non-null   float64
 11  Consolidation               70622 non-null   float64
 12  Pneumonia                   27608 non-null   float64
 13  Atelectasis              

**Columns normalization**

In [5]:
df_xray_clean = df_xray_raw.copy()

df_xray_clean.columns = (df_xray_clean.columns.str.lower()
                                              .str.strip()
                                              .str.replace(" ", "_")
                                              .str.replace("/","_"))

In [6]:
print(df_xray_clean.isnull().sum())

path                               0
sex                                0
age                                0
frontal_lateral                    0
ap_pa                          32387
no_finding                    201033
enlarged_cardiomediastinum    178575
cardiomegaly                  177211
lung_opacity                  105636
lung_lesion                   211470
edema                         137458
consolidation                 152792
pneumonia                     195806
atelectasis                   154971
pneumothorax                  144480
pleural_effusion               90203
pleural_other                 216922
fracture                      211220
support_devices               100197
dtype: int64


*=> We can observe a large number of null values, which is normal because most of the columns represent the range of possible pathologies for a patient*

**Descriptive statistics**

In [7]:
print(df_xray_clean.describe())

                 age  no_finding  enlarged_cardiomediastinum  cardiomegaly  \
count  223414.000000     22381.0                44839.000000  46203.000000   
mean       60.430653         1.0                   -0.035795      0.409346   
std        17.820925         0.0                    0.718442      0.769323   
min         0.000000         1.0                   -1.000000     -1.000000   
25%        49.000000         1.0                   -1.000000      0.000000   
50%        62.000000         1.0                    0.000000      1.000000   
75%        74.000000         1.0                    0.000000      1.000000   
max        90.000000         1.0                    1.000000      1.000000   

        lung_opacity   lung_lesion         edema  consolidation     pneumonia  \
count  117778.000000  11944.000000  85956.000000   70622.000000  27608.000000   
mean        0.848911      0.644508      0.456769      -0.183498     -0.461134   
std         0.472571      0.691607      0.741785      

In [8]:

nb_duplicated = df_xray_clean.duplicated().sum()
print(f"The dataset contains {nb_duplicated} duplicates")

The dataset contains 0 duplicates


**Data quality**

In [9]:
print(df_xray_clean.groupby("sex")["sex"].count())


sex
Female      90777
Male       132636
Unknown         1
Name: sex, dtype: int64


In [ ]:
# Remove values ​​that are not "male" or "female"
list_sex = ["male","female"]
df_xray_clean["sex"] = df_xray_clean["sex"].str.lower().str.strip()
df_xray_clean = df_xray_clean.drop(df_xray_clean[~df_xray_clean["sex"].isin(list_sex)].index)
print(df_xray_clean.groupby("sex")["sex"].count())

# Creation of a new column "sex_numeric" that is : 1 for "male" and 0 for "female" 
dict_sex = {"male":1,"female":0}
df_xray_clean["sex_numeric"] = df_xray_clean["sex"].map(dict_sex)

sex
female     90777
male      132636
Name: sex, dtype: int64


In [11]:
for col in df_xray_clean.columns :
    print(df_xray_clean[col].value_counts())

path
CheXpert-v1.0-small/train/patient00001/study1/view1_frontal.jpg    1
CheXpert-v1.0-small/train/patient00002/study2/view1_frontal.jpg    1
CheXpert-v1.0-small/train/patient00002/study1/view1_frontal.jpg    1
CheXpert-v1.0-small/train/patient00002/study1/view2_lateral.jpg    1
CheXpert-v1.0-small/train/patient00003/study1/view1_frontal.jpg    1
                                                                  ..
CheXpert-v1.0-small/train/patient64537/study2/view1_frontal.jpg    1
CheXpert-v1.0-small/train/patient64537/study1/view1_frontal.jpg    1
CheXpert-v1.0-small/train/patient64538/study1/view1_frontal.jpg    1
CheXpert-v1.0-small/train/patient64539/study1/view1_frontal.jpg    1
CheXpert-v1.0-small/train/patient64540/study1/view1_frontal.jpg    1
Name: count, Length: 223413, dtype: int64
sex
male      132636
female     90777
Name: count, dtype: int64
age
90    7579
61    5372
65    5098
66    5098
58    5075
      ... 
21    1279
23    1229
19    1167
18     766
0        3
Name:

**Save clean data to parquet**


In [12]:
df_xray_clean.to_csv("../../../data/processed/data_clean.csv", index=False)